# 模型部署与服务化：从 Checkpoint 到 vLLM / SGLang API

> 前面的章节已经解释了 Serving System 为什么需要 Continuous Batching、PagedAttention、Prefix Cache、Chunked Prefill、PD 分离。
>
> 这一章不再重复解释这些原理。
>
> 现在只做一件事：
>
> **把一个 Hugging Face Checkpoint 真正变成可调用、可压测、可排错的在线服务。**
>
> 目标是读完后你能自己完成：
>
> ```text
> 选择模型
> → 安装环境
> → 启动 vLLM
> → curl / OpenAI SDK 调用
> → Streaming
> → 调整显存和并发
> → 启动 SGLang
> → 对比两套参数
> → 多 GPU TP
> → 量化模型
> → Benchmark
> → Troubleshooting
> → 理解 PD / KV Transfer 的部署拓扑
> ```
>
> 这里用小模型示范启动方式，避免教程把“概念正确”建立在 8 张 H100 上。真正部署大模型时，命令结构相同，主要变化是模型名、并行度、Context Length 和显存参数。

先明确一件事：Hugging Face Checkpoint 本身还不是服务。

## 1. 从 Checkpoint 到 API，中间多了什么？

在 Notebook 里：

```python
model.generate(...)
```

看起来只有模型。

在线服务实际多了一整层系统：

```text
Hugging Face Checkpoint
        ↓
Tokenizer / Chat Template
        ↓
Inference Engine
├─ Scheduler
├─ KV Cache Manager
├─ Kernel Backend
└─ Distributed Runtime
        ↓
HTTP Server
        ↓
OpenAI-Compatible API
        ↓
Client / Gateway / Application
```

vLLM 和 SGLang 的价值，不是“把 `generate()` 包一层 HTTP”。

它们把前一章讲过的大量 Serving Mechanism 真正实现出来。

## 2. 部署前先检查四件事

真正开始前，先回答：

### 2.1 GPU 能不能放下模型？

粗估：

```text
7B BF16 ≈ 14 GB 权重
7B INT8 ≈ 7 GB
7B INT4 ≈ 3.5 GB
```

还要额外留：

- KV Cache
- CUDA / framework workspace
- graph capture
- temporary buffers

所以“显存刚好等于权重大小”通常不够。

### 2.2 模型最大 Context 多长？

Context Length 越大，KV Cache 上限越大。

### 2.3 模型有没有正确 Chat Template？

Instruct / Chat 模型需要匹配训练格式。

### 2.4 当前硬件支持哪些精度 / Kernel？

例如：

- BF16
- FP16
- FP8
- AWQ / GPTQ
- 特定 Attention Backend

部署参数一定要结合 GPU 架构。

## 3. vLLM：先启动最小 OpenAI-Compatible Server

当前 vLLM 的基本入口是：

```bash
vllm serve <MODEL>
```

例如官方 Quickstart 常用：

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct
```

默认会启动在：

```text
http://localhost:8000
```

先不要加一堆高级参数。

第一步只确认三件事：

```text
模型能下载
↓
权重能加载
↓
API 能返回
```

### 安装环境

实际安装方式要根据 CUDA / ROCm / TPU / Apple Silicon 选择。

对于常规 Linux + NVIDIA 环境，建议先单独创建 Python 环境，再按 vLLM 官方安装方式安装。

```bash
python -m venv .venv
source .venv/bin/activate

pip install -U pip
pip install vllm openai
```

如果遇到 CUDA / PyTorch ABI 问题，不要先改业务代码。

先确认：

```bash
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
nvidia-smi
vllm --help
```

部署排错的第一原则是：**先判断环境层，还是模型层。**

## 4. 第一次启动 vLLM

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct   --host 0.0.0.0   --port 8000
```

服务起来后先检查模型列表：

```bash
curl http://localhost:8000/v1/models
```

这一步成功，才继续发 Chat 请求。

### curl 调用 Chat Completions

```bash
curl http://localhost:8000/v1/chat/completions   -H "Content-Type: application/json"   -d '{
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
      {"role": "user", "content": "用三句话解释 KV Cache"}
    ],
    "temperature": 0.2,
    "max_tokens": 128
  }'
```

如果这里报 Chat Template 错误，说明：

> 模型 / Tokenizer 没有可用的 Chat Template，或者你应该使用 Completion API / 手动指定模板。

## 5. 用 OpenAI Python SDK 调 vLLM

OpenAI-compatible 的价值是：上层应用不需要知道底下是 vLLM 还是远程模型。

```python
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {"role": "user", "content": "What is PagedAttention?"}
    ],
)

print(response.choices[0].message.content)
```

以后接 LangChain、Agent Framework、自己的 Web 后端，本质上也是改 `base_url`。

In [ ]:
# 这个 Cell 只展示请求对象，不会真的访问本地服务。
request_example = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {"role": "user", "content": "What is PagedAttention?"}
    ],
    "temperature": 0.2,
    "max_tokens": 128,
}
print(request_example)

## 6. Streaming：为什么在线聊天要流式返回？

非 Streaming：

```text
模型生成 500 tokens
↓
全部生成完
↓
一次性返回
```

用户可能等很久才看到任何内容。

Streaming：

```text
token chunk
↓
立刻返回
token chunk
↓
继续返回
```

所以用户体验更接近：

```text
TTFT 之后开始持续看到输出
```

OpenAI SDK：

```python
stream = client.chat.completions.create(
    model=MODEL,
    messages=[...],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
```

## 7. `generation_config.json`：为什么我的 Temperature 和预期不一样？

vLLM 会读取模型仓库中的 Generation Config。

这意味着某些模型可能自带：

```text
temperature
top_p
repetition_penalty
```

等默认值。

如果你做 Benchmark，希望统一由 vLLM 默认策略控制，可以关注：

```bash
--generation-config vllm
```

这和第 25 章直接对应：

> 评测前必须知道 Sampling Config 从哪里来。

“同一个模型、同一套 API 参数”也可能因为模型自带 Generation Config 而表现不同。

## 8. OOM：第一反应应该调什么？

部署最常见报错之一就是：

```text
CUDA out of memory
```

先区分四类显存：

```text
Model Weights
KV Cache
Runtime / Workspace
CUDA Graph / Temporary Buffers
```

常见调节方向：

### 降低最大 Context

```bash
--max-model-len 8192
```

Context 上限过大，会压缩 KV Cache 可用空间。

### 调整 GPU Memory Utilization

```bash
--gpu-memory-utilization 0.90
```

### 降低并发上限

关注类似：

```text
max-num-seqs
max-num-batched-tokens
```

### 换低精度 / 量化模型

```text
BF16 → FP8 / AWQ / GPTQ
```

所以 OOM 不是一句“模型太大”就结束。

## 9. 为什么 `max-model-len` 会影响并发？

假设每个请求理论上允许 128k Context。

Serving Engine 必须为可能的 KV Cache 需求做容量规划。

如果真实业务 99% 请求只有 8k：

```text
把 max-model-len 设得极大
```

可能让单请求上限很漂亮，但高并发容量变差。

这就是实际部署里常见的 trade-off：

```text
单请求最大 Context
vs
同时能服务多少请求
```

所以参数不是“越大越好”。

## 10. Tensor Parallel：一张 GPU 放不下怎么办？

如果一个模型需要多 GPU：

```bash
vllm serve <MODEL>   --tensor-parallel-size 4
```

对应上一章的 **TP — Tensor Parallel**：

```text
同一层的大矩阵
→ 切到 4 张 GPU
→ 每层需要跨卡通信
```

所以 TP 增大通常意味着：

```text
单卡显存压力下降
但通信成本上升
```

不要把“8 GPU”自动理解为“吞吐 8 倍”。

## 11. 量化模型怎么启动？

很多预量化模型可以直接换模型名。

例如概念上：

```bash
vllm serve <AWQ_MODEL>
```

或者：

```bash
vllm serve <FP8_MODEL>
```

但是否支持取决于：

- vLLM 版本
- GPU 架构
- 模型格式
- Quantization Kernel

所以看到报错：

```text
unsupported quantization method
kernel not available
```

先检查兼容矩阵，而不是把量化权重强行转格式。

## 12. SGLang：启动方式和 vLLM 有什么不同？

SGLang 同样可以启动 OpenAI-compatible API。

常见入口之一：

```bash
python -m sglang.launch_server   --model-path Qwen/Qwen3-8B   --host 0.0.0.0   --port 30000
```

较新的 SGLang 环境也提供 `sglang serve` 形式。

服务通常可通过：

```text
http://localhost:30000/v1
```

访问。

先用和 vLLM 完全相同的 OpenAI SDK 请求去验证。

这样对比两个 Engine 时，上层 Prompt / Client 不变，只换 Serving Backend。

### SGLang 多 GPU Tensor Parallel

```bash
python -m sglang.launch_server   --model-path Qwen/Qwen3-8B   --tp 4   --port 30000
```

或者对应版本支持的完整参数名。

这里再次对应上一章：

```text
TP
→ 模型层内部切分
→ 需要跨 GPU 通信
```

所以部署命令里的 `--tp` 不应该只是“一个神秘参数”。

## 13. SGLang 为什么经常和 RadixAttention 一起出现？

上一章已经讲过 RadixAttention 的位置：

```text
相同 / 重叠 Prefix
↓
Radix Tree 管理 Prefix KV
↓
复用已经计算的 KV
```

所以 SGLang 在多轮对话、Agent、共享 System Prompt 等 workload 下，经常强调 Prefix Reuse。

部署时真正应该观察的是：

```text
Cache hit rate
TTFT
Throughput
Memory
```

而不是只记“RadixAttention 是 SGLang 的特色”。

## 14. vLLM vs SGLang：不要问“谁永远更快”

更好的比较方式是固定：

```text
同一个 Model
同一个 Precision
同一张 GPU / 同一集群
同一个 Input/Output Length 分布
同一个 Concurrency
同一个 Sampling Config
同一个 SLO
```

然后比较：

| 指标 | vLLM | SGLang |
|---|---:|---:|
| TTFT | | |
| TPOT | | |
| Throughput | | |
| GPU Memory | | |
| P99 | | |
| Prefix-heavy workload | | |

不同 workload，胜负可能不同。

所以“框架选型”是 Benchmark 问题，不是信仰问题。

## 15. 一个最小 Benchmark 应该怎么做？

不要只跑：

```bash
curl
```

然后觉得服务已经“性能不错”。

至少准备：

```text
固定请求集
固定 input length
固定 output length
不同 concurrency
记录 TTFT / TPOT / throughput
```

例如：

```text
Concurrency = 1, 8, 32, 128
Prompt = 1k
Output = 256
```

观察随着并发增加：

- Throughput 怎么涨？
- TPOT 什么时候开始恶化？
- P99 什么时候突然上升？
- GPU Memory 是否逼近上限？

这就是系统容量曲线。

In [ ]:
# 一个 benchmark 记录表的最小结构
benchmark_rows = [
    {"concurrency": 1, "ttft_ms": None, "tpot_ms": None, "throughput": None},
    {"concurrency": 8, "ttft_ms": None, "tpot_ms": None, "throughput": None},
    {"concurrency": 32, "ttft_ms": None, "tpot_ms": None, "throughput": None},
    {"concurrency": 128, "ttft_ms": None, "tpot_ms": None, "throughput": None},
]

for row in benchmark_rows:
    print(row)

## 16. Troubleshooting：服务起不来时按层排查

### Layer 1 — Environment

```text
CUDA
Driver
PyTorch
Python
vLLM / SGLang wheel
```

### Layer 2 — Model

```text
模型文件下载完整？
Config 能识别？
Tokenizer 正确？
Trust Remote Code？
Chat Template？
```

### Layer 3 — Memory

```text
权重
max-model-len
KV Cache
Concurrency
CUDA Graph
```

### Layer 4 — Distributed

```text
TP 数量
NCCL
NVLink / PCIe
Host Network
Rank 启动
```

### Layer 5 — API

```text
Model name
base_url
endpoint
JSON payload
streaming
auth
```

不要一看到错误就随机加参数。

先判断它属于哪一层。

## 17. 为什么服务能跑，但输出质量很怪？

常见原因：

### Chat Template 不对

Instruct Model 被当 Base Model 使用。

### Generation Config 不一致

Temperature / Top-p 被模型默认配置覆盖。

### Tokenizer 不匹配

本地路径或权重 / tokenizer 版本混用。

### Quantization 质量下降

低比特导致特定任务回退。

### Context Truncation

Prompt 超过限制，被截掉关键上下文。

所以：

> “API 返回 200”只代表服务活着，不代表模型行为正确。

## 18. 为什么吞吐很高，但用户觉得很慢？

可能是：

```text
Concurrency 拉太高
↓
Throughput 上升
↓
每个请求 TPOT / TTFT 变差
```

也可能是：

```text
长 Prefill
↓
Decode 被阻塞
↓
P99 ITL 爆炸
```

所以生产系统需要 SLO：

```text
TTFT P95 < ?
TPOT P95 < ?
P99 latency < ?
```

然后在 SLO 约束下最大化 Throughput。

这比单独追峰值 tok/s 更接近真实业务。

## 19. PD 分离真正部署时是什么样？

上一章讲的是概念，现在把它画成部署拓扑：

```text
                ┌──────────────┐
Request ───────→│   Router     │
                └──────┬───────┘
                       ↓
                ┌──────────────┐
                │ Prefill Pool │
                └──────┬───────┘
                       │ KV Cache
                       │ Transfer
                       ↓
                ┌──────────────┐
                │ Decode Pool  │
                └──────┬───────┘
                       ↓
                    Stream
```

这里真正困难的是中间箭头：

> **KV 怎么从 Prefill Worker 高效传到 Decode Worker？**

于是工程里会看到：

- KV Connector
- RDMA
- NIXL / Mooncake 一类传输或存储方案
- remote / disaggregated KV cache
- routing / load balancing

PD 分离不是“打开一个参数就自动更快”。

它适合：

- Prefill / Decode workload 明显不平衡
- 需要独立控制 TTFT 和 TPOT
- 集群规模足够大
- KV Transfer 成本可控

小模型、单卡、本地服务通常没有必要一上来就做 PD。

所以看到厂商强调 PD 分离时，应该继续问：

> **他们的 workload 为什么值得把系统复杂度提高到这一层？**

## 20. 从单机走到生产集群，还缺什么？

Inference Engine 只是核心。

真正生产系统还需要：

```text
API Gateway
Authentication
Rate Limit
Load Balancer
Autoscaling
Metrics
Tracing
Logging
Canary
Model Versioning
Health Check
Timeout / Retry
Admission Control
```

这些已经开始进入 Platform / Infra，而不只是“LLM Kernel”。

招聘 JD 如果写：

```text
Kubernetes
Ray
Prometheus
Grafana
NCCL
RDMA
Autoscaling
```

说明岗位可能已经从 Inference Engine 延伸到 Serving Platform。

## 21. 你现在应该能读懂一条推理 JD

例如：

> 熟悉 vLLM / SGLang，理解 Continuous Batching、PagedAttention、Prefix Cache、Chunked Prefill、PD Disaggregation，具备 Tensor Parallel、NCCL、CUDA Graph、FlashAttention 优化经验。

读到这里应该能拆成：

```text
vLLM / SGLang
→ Engine

Continuous Batching
→ Scheduler

PagedAttention / Prefix Cache
→ KV Memory

Chunked Prefill
→ Prefill / Decode 调度

PD Disaggregation
→ Cluster topology + KV Transfer

Tensor Parallel / NCCL
→ Distributed

CUDA Graph / FlashAttention
→ Kernel / Runtime
```

这就是 Part 3 最终想建立的系统地图。

## 小结

自己部署 vLLM / SGLang 时，按这条链走：

```text
1. 先用小模型跑通
2. 验证 /v1/models
3. 验证 Chat Completions
4. 验证 Streaming
5. 固定 Generation Config
6. 再调 Context / Memory / Concurrency
7. 再做 Quantization
8. 再做 Multi-GPU TP
9. 固定 Workload 做 Benchmark
10. 最后才考虑 PD / KV Transfer / 大规模集群
```

不要一开始就堆高级参数。

一个系统能被理解，通常是因为你知道每个参数在解决哪一个具体问题。

## 作业

1. 在一张可用 GPU 上启动一个最小 vLLM 服务，并用 `/v1/models` 验证。
2. 分别用 curl 和 OpenAI Python SDK 完成一次 Chat Completion。
3. 开启 Streaming，记录 TTFT 和完整请求耗时。
4. 把 `max-model-len` 改成两个不同值，观察可用 KV Cache / 并发变化。
5. 用相同模型启动 SGLang，固定 Prompt 和 Sampling 参数，对比 vLLM。
6. 如果有多 GPU，尝试 `TP=2`，记录显存和性能变化。
7. 画出你自己的 `Gateway → Prefill → KV Transfer → Decode` 拓扑，并说明什么时候你认为值得做 PD 分离。

## 参考与版本提醒

Serving 框架变化很快。命令和参数应以你实际安装版本的官方文档为准。

本章使用的核心入口与当前文档保持一致：

- vLLM：`vllm serve <model>`，OpenAI-compatible API 默认常见端口为 8000。
- SGLang：`python -m sglang.launch_server --model-path <model>` 或对应版本的 `sglang serve`，OpenAI-compatible API 常见端口为 30000。
- 多 GPU：两者都支持 Tensor Parallel，但具体参数名请以当前版本 CLI `--help` 为准。

真正稳定的知识不是某个 flag，而是前面建立的 Serving Mental Model。